In [1]:
import parsl, random
from parsl import python_app
from parsl.providers import LocalProvider
from parsl.executors import HighThroughputExecutor
from parsl.config import Config

import json, csv, time, os, sys
from moldesign.simulate.functions import generate_inchi_and_xyz, relax_structure
from moldesign.simulate.specs import get_qcinput_specification
from moldesign.store.models import MoleculeData
from moldesign.store.recipes import apply_recipes
from rdkit import Chem
import logging, qcengine

In [2]:
def load_json_file(file_path):
    data = []
    with open(file_path) as file:
         for line in file:
                try:
                    item = json.loads(line)
                    data.append(item)
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON: {e}")
    return data

In [4]:
file_path = "./dataset/training-data.json"
json_data = load_json_file(file_path)

dataset = []
smiles_list = []
inchi_list = []
ip_list = []

for item in json_data:
    if "identifier" in item and "oxidation_potential" in item:
                smiles = item["identifier"]["smiles"]
                dataset_item = {
                        "smiles": item["identifier"]["smiles"],
                        "inchi": item["identifier"]["inchi"],
                        "ip": item["oxidation_potential"]["xtb-vacuum"],
                }
                smiles_list.append(item["identifier"]["smiles"])
                inchi_list.append(item["identifier"]["inchi"])
                ip_list.append(item["oxidation_potential"]["xtb-vacuum"])
                dataset.append(dataset_item)

print(len(smiles_list), len(ip_list))

4511 4511


In [5]:
from tensorflow.python.keras import callbacks as cb
from typing import List, Any, Optional, Tuple, Dict, Union
from moldesign.utils.conversions import convert_string_to_dict
from moldesign.utils.callbacks import LRLogger, EpochTimeLogger, TimeLimitCallback
from moldesign.score.nfp import make_data_loader, ReduceAtoms

import nfp, random
import tensorflow as tf
import numpy as np
import pickle as pkl

In [6]:
x_tmp = []
y_tmp = []
for smiles in smiles_list:
    x_tmp.append(convert_string_to_dict(smiles))
    y_tmp.append(float(ip_list[smiles_list.index(smiles)]))
mol_dicts = np.array(x_tmp)
max_size = max(len(x['atom']) for x in mol_dicts)
y = np.array(y_tmp)

random_state = 1
validation_split = 0.1

# Make the training and validation splits
rng = np.random.RandomState(random_state)
train_split = rng.rand(len(x_tmp)) > validation_split
train_X = mol_dicts[train_split]
train_y = y[train_split]
valid_X = mol_dicts[~train_split]
valid_y = y[~train_split]

print(len(train_X), len(valid_X))

4062 449


In [7]:
# Make the loaders
batch_size = 128

steps_per_epoch = len(train_X) // batch_size
train_loader = make_data_loader(train_X, train_y, repeat=True, batch_size=batch_size, max_size=max_size, drop_last_batch=True, shuffle_buffer=32768)
valid_steps = len(valid_X) // batch_size
valid_loader = make_data_loader(valid_X, valid_y, batch_size=batch_size, max_size=max_size, drop_last_batch=True)

custom_objects = nfp.custom_objects.copy()
custom_objects['ReduceAtoms'] = ReduceAtoms

In [ ]:
# Load Model
model = tf.keras.models.load_model("./model.h5", custom_objects=custom_objects, compile=True)
learning_rate = 1e-3

checkpoint_path = "./training/cp.ckpt"
chekcpoint_dir = os.path.dirname(checkpoint_path)

cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path, save_weights_only=True, verbose=1)

try:
    scaler_layer = model.get_layer('scale')
    outputs = np.array(y_tmp)
    scaler_layer.set_weights([outputs.std()[None, None], outputs.mean()[None]])
except ValueError:
    pass

model.compile(
            tf.optimizers.Adam(learning_rate),
            'mean_squared_error',
            metrics=['mean_absolute_error'],
            steps_per_execution=1
        )

num_epochs = 10

history = model.fit(
            train_loader,
            epochs=num_epochs,
            shuffle=False,
            #verbose=False,
            steps_per_epoch=len(train_X),
            validation_data=valid_loader,
            validation_steps=valid_steps,
            validation_freq=1,
            callbacks=[cp_callback]
        )

# save model
date = int(time.time() * 1000)
model_name = './training/networks/model-' + str(date) + '.h5'
model.save(model_name)

print("Finish model fit")
print(history)

Epoch 1/10
4062/4062 [==============================] - ETA: 0s - loss: 0.1070 - mean_absolute_error: 0.2020
Epoch 1: saving model to ./training/cp.ckpt


2023-07-07 00:14:15.457585: W tensorflow/core/kernels/data/cache_dataset_ops.cc:768] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


4062/4062 [==============================] - 1977s 484ms/step - loss: 0.1070 - mean_absolute_error: 0.2020 - val_loss: 0.1142 - val_mean_absolute_error: 0.2019
Epoch 2/10
4062/4062 [==============================] - ETA: 0s - loss: 0.0682 - mean_absolute_error: 0.1591
Epoch 2: saving model to ./training/cp.ckpt


2023-07-07 00:47:06.952878: W tensorflow/core/kernels/data/cache_dataset_ops.cc:768] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


4062/4062 [==============================] - 1971s 485ms/step - loss: 0.0682 - mean_absolute_error: 0.1591 - val_loss: 0.1161 - val_mean_absolute_error: 0.2017
Epoch 3/10
4062/4062 [==============================] - ETA: 0s - loss: 0.0464 - mean_absolute_error: 0.1295
Epoch 3: saving model to ./training/cp.ckpt


...

4062/4062 [==============================] - 1975s 486ms/step - loss: 0.0170 - mean_absolute_error: 0.0762 - val_loss: 0.1303 - val_mean_absolute_error: 0.2228
Epoch 9/10
4062/4062 [==============================] - ETA: 0s - loss: 0.0387 - mean_absolute_error: 0.1131
Epoch 9: saving model to ./training/cp.ckpt


2023-07-07 04:37:14.181079: W tensorflow/core/kernels/data/cache_dataset_ops.cc:768] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


4062/4062 [==============================] - 1977s 487ms/step - loss: 0.0387 - mean_absolute_error: 0.1131 - val_loss: 0.1249 - val_mean_absolute_error: 0.2081
Epoch 10/10
4062/4062 [==============================] - ETA: 0s - loss: 0.0770 - mean_absolute_error: 0.1431
Epoch 10: saving model to ./training/cp.ckpt


2023-07-07 05:10:08.036371: W tensorflow/core/kernels/data/cache_dataset_ops.cc:768] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


4062/4062 [==============================] - 1973s 486ms/step - loss: 0.0770 - mean_absolute_error: 0.1431 - val_loss: 0.1754 - val_mean_absolute_error: 0.2382


In [ ]:
# Load Model
model = tf.keras.models.load_model("./model.h5", custom_objects=custom_objects, compile=True)
learning_rate = 1e-3

weights = []
for v in model.get_weights():
    v = np.array(v)
    if np.isnan(v).any():
        raise ValueError('Found some NaN weights.')
    weights.append(v)

weights_json_str = model.to_json()
model_json = tf.keras.models.model_from_json(weights_json_str, custom_objects=custom_objects)

checkpoint_path = "./training/cp.ckpt"
chekcpoint_dir = os.path.dirname(checkpoint_path)

cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path, save_weights_only=True, verbose=1)

try:
    scaler_layer = model.get_layer('scale')
    outputs = np.array(y_tmp)
    scaler_layer.set_weights([outputs.std()[None, None], outputs.mean()[None]])
except ValueError:
    pass

num_epochs = 3

model_json.compile(
            tf.optimizers.Adam(learning_rate),
            'mean_squared_error',
            metrics=['mean_absolute_error'],
            steps_per_execution=1
        )

history_json = model_json.fit(
            train_loader,
            epochs=num_epochs,
            shuffle=False,
            #verbose=False,
            steps_per_epoch=len(train_X),
            validation_data=valid_loader,
            validation_steps=valid_steps,
            validation_freq=1,
            callbacks=[cp_callback]
        )

print("Finish model_json fit")
print(history_json)

model.compile(
            tf.optimizers.Adam(learning_rate),
            'mean_squared_error',
            metrics=['mean_absolute_error'],
            steps_per_execution=1
        )

history = model.fit(
            train_loader,
            epochs=num_epochs,
            shuffle=False,
            #verbose=False,
            steps_per_epoch=len(train_X),
            validation_data=valid_loader,
            validation_steps=valid_steps,
            validation_freq=1,
            callbacks=[cp_callback]
        )

# save model
date = int(time.time() * 1000)
model_name = './training/networks/model-' + str(date) + '.h5'
model.save(model_name)

print("Finish model fit")
print(history)

Epoch 1/3
4062/4062 [==============================] - ETA: 0s - loss: 466.2358 - mean_absolute_error: 1.3310
Epoch 1: saving model to ./training/cp.ckpt


2023-07-13 14:50:25.091690: W tensorflow/core/kernels/data/cache_dataset_ops.cc:768] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


4062/4062 [==============================] - 1895s 464ms/step - loss: 466.2358 - mean_absolute_error: 1.3310 - val_loss: 23.6275 - val_mean_absolute_error: 4.1840
Epoch 2/3
3946/4062 [============================>.] - ETA: 54s - loss: 6.5449 - mean_absolute_error: 1.7954